In [1]:
import numpy as np
import nibabel as nib
import os
import glob
import re

In [2]:
gica_output = 'CIMT_data/no_wm/LLstim_ICA_19c'

In [3]:
agg_comp = nib.load(os.path.join(gica_output, gica_output.split('/')[-1] + '_agg__component_ica_.nii'))
agg_comp_data = agg_comp.get_fdata()
agg_comp_data.shape

(96, 96, 25, 19)

In [4]:
agg_comp_stan = np.zeros(shape=agg_comp_data.shape)
for ic in range(agg_comp_data.shape[3]):
    mean = np.mean(agg_comp_data[:, :, :, ic])
    std = np.std(agg_comp_data[:, :, :, ic])
    agg_comp_stan[:, :, :, ic] = (agg_comp_data[:, :, :, ic] - mean) / std

In [5]:
agg_comp_stan_nii = nib.Nifti1Image(agg_comp_stan, agg_comp.affine, header=agg_comp.header)
nib.save(agg_comp_stan_nii, os.path.join(gica_output, gica_output.split('/')[-1] + '_agg__component_ica_standardized_.nii'))

In [12]:
wkdir = '/Users/fei/Desktop/Harris_Lab/dFNC/CIMT_data/ICA/All_infomax/MELODIC_Train'
z_scored_path = '/Users/fei/Desktop/Harris_Lab/dFNC/CIMT_data/ICA/All_infomax/MELODIC_Train_Z'


train_list = sorted(os.listdir(wkdir))

for folder in train_list:
    t = os.path.join(wkdir, folder)
    rat_id = t[t.rfind('/')+1:-12]
    if t.endswith('ica'):
        output = glob.glob(t + '/*.ica')[0]
        ica = nib.load(glob.glob(output + '/melodic_IC.nii.gz')[0])
        ica_data = ica.get_fdata()
        print(ica_data.shape)
        print(np.mean(ica_data), np.std(ica_data))
        # affine = ica.affine
        # header = ica.header

        # for i in range(ica_data.shape[3]):
        #     ic_map = ica_data[:, :, :, i]
        #     ica_data[:, :, :, i] = (ic_map - np.mean(ic_map)) / np.std(ic_map)

        # z_ica = nib.Nifti1Image(ica_data, affine, header)
        # nib.save(z_ica, os.path.join(z_scored_path, rat_id+'_z_scored.nii'))

(96, 96, 25, 49)
0.013847578258371726 0.3975387906932875
(96, 96, 25, 58)
0.013559242386222865 0.38029205469075206
(96, 96, 25, 55)
0.013596091816747879 0.3882513212859946
(96, 96, 25, 54)
0.015288702974177384 0.41262130393283614
(96, 96, 25, 52)
0.014041908093051654 0.4072118347039454
(96, 96, 25, 58)
0.014393946653498512 0.404893618667597
(96, 96, 25, 50)
0.016254112918872216 0.4358766526066409
(96, 96, 25, 50)
0.014519341440539138 0.3994536353004492
(96, 96, 25, 56)
0.014148022835523745 0.4000072917414676
(96, 96, 25, 54)
0.013175169724555682 0.387925736978212
(96, 96, 25, 51)
0.013477888443133145 0.3943048271570395
(96, 96, 25, 56)
0.01445365348179639 0.3788628506131032
(96, 96, 25, 53)
0.01430504847455854 0.3985273699380277
(96, 96, 25, 55)
0.012782205020261395 0.37047341359586516
(96, 96, 25, 53)
0.013662512471838688 0.4070357963348718
(96, 96, 25, 58)
0.013676438595026125 0.4169011872523701
(96, 96, 25, 57)
0.01772323717519847 0.4277402082517938
(96, 96, 25, 56)
0.01492113121132

In [8]:
# # Load MELODIC ICA results and mixing matrices of training rats
# wkdir = '/Users/fei/Desktop/Harris_Lab/dFNC/CIMT_data/ICA/All_infomax/MELODIC_Train'
# raw_maps_dir = '/Users/fei/Desktop/Harris_Lab/dFNC/CIMT_data'

# train_list = sorted(os.listdir(wkdir))
# raw_maps_list = sorted(os.listdir(raw_maps_dir))

# ### Number of components in each of the 18 training datasets
# # 49, 58, 55, 54, 
# # 52, 58, 50, 50, 
# # 56, 54, 51, 56,
# # 53, 55,
# # 53, 58,
# # 57, 56

# tp = 250 # number of timepoints
# days = ['07d', '21d', '49d'] # folders for each day for the raw data

# datasets = [] # the independent components for each training rat, each array: (X, Y, Z, components)
# mix_matrices = [] # the corresponding mixing matrices for each ICA result, each matrix: (tp, components)
# raw_maps = [] # lists containing: the corresponding original fMRI data for each training rat, each array: (X, Y, Z, tp)
#                                 # the nii affine,
#                                 # the nii header,
#                                 # and the rat id

# for folder in train_list:
#     t = os.path.join(wkdir, folder)
#     if t.endswith('ica'):
#         output = glob.glob(t + '/*.ica')[0]
#         ica = nib.load(glob.glob(output + '/melodic_IC.nii.gz')[0])
#         ica_data = ica.get_fdata()
#         datasets.append(ica_data)
#         with open(glob.glob(output + '/melodic_mix')[0], 'r') as f:
#             mix = [float(re.sub(r'[^-0-9\.]', '', n)) if 'e' not in n else float(re.sub(r'[^-0-9\.]', '', n.split('e')[0])) * (10 ** int(n.split('e')[1].replace('0', ''))) 
#                    for n in f.read().split('  ')[:-1]] # some numbers are written in scientific notation and the last element is just whitespace
#             mix = np.array(mix) 
#             print(mix[-1]) # sanity check
#             print(len(mix) / tp) # another sanity check
#             mix = np.reshape(mix, (tp, -1))
#             print(mix.shape)
#             print(mix[0, :])
#             mix_matrices.append(mix)

        
#         rat_id = t[t.rfind('/')+1:-12]
#         for d in days:
#             if d in rat_id:
#                 print(glob.glob(f'{raw_maps_dir}/{d}/*/*/*/{rat_id}.nii')[0])
#                 raw = nib.load(glob.glob(f'{raw_maps_dir}/{d}/*/*/*/{rat_id}.nii')[0])
#                 raw_data = raw.get_fdata()
#                 raw_maps.append([raw_data, raw.affine, raw.header, rat_id])
#                 print(raw_data.shape)

# print(len(datasets) == len(mix_matrices) == len(raw_maps))

0.7817869231
49.0
(250, 49)
[-5.26649949e-01 -1.46596404e+00 -8.15731026e-01 -1.31948525e+00
 -1.76756115e-01  2.70602419e+00  5.08130084e-01  8.96799113e-01
  8.59668213e-01  4.85479211e-04  9.21428785e-01  2.15637658e+00
 -1.06594181e+00 -1.97846641e+00  1.73197735e+00  1.15277273e-01
 -8.08662318e-01  5.93180933e+00 -9.65643361e-01 -1.19091485e+00
 -1.46298113e+00  1.54147742e+00 -1.10590255e+00 -3.10879732e+00
 -3.00036041e-02  7.55713599e-01 -3.05356446e-01 -9.63868129e-01
 -2.10445094e+00  6.69245755e-01  7.94732836e-01  4.05565454e-01
 -1.85677155e-01  3.29039472e-01 -8.37803290e-01 -7.87359483e-02
  2.81041821e-01  1.04963091e+00 -2.81958931e+00 -2.57358745e-01
  1.29005262e+00  4.85463585e-01  1.62512824e+00  8.42503533e-01
  2.92346215e-01  2.39530354e+00 -2.03821155e+00 -8.17287281e-02
 -1.63960358e+00]
/Users/fei/Desktop/Harris_Lab/dFNC/CIMT_data/07d/07d_nii/All/CCI_002_07d/CCI_none_07d_002.nii
(96, 96, 25, 250)
1.366140995
58.0
(250, 58)
[ 0.07292987 -1.1638717   0.1003465

In [9]:
# print(datasets[0].shape) # should be X, Y, Z, #ICs
# print(raw_maps[0][0].shape) # should be X, Y, Z, tp

In [10]:
# # Z-score ICA maps
# z_path = '/Users/fei/Desktop/Harris_Lab/dFNC/CIMT_data/ICA/All_infomax/MELODIC_Train_Z'

# for i in range(len(datasets)):
#     A = mix_matrices[i] # A: (tp, components)
#     W = np.linalg.inv(A.T @ A) @ A.T # W: (components, tp)
#     P = np.identity(W.shape[1]) - (W.T @ W) # P: (tp, tp)

#     raw_4d = raw_maps[i][0] # raw_4d: (X, Y, Z, tp)
#     xdim, ydim, zdim, time = raw_4d.shape
#     raw_2d = raw_4d.reshape(-1, time).T # raw_2d: (tp, X*Y*Z)
#     print(raw_2d.shape) # sanity check
#     print(np.array_equal(raw_4d, raw_2d.T.reshape(xdim, ydim, zdim, time))) # sanity check

#     n_2d = P @ raw_2d # n_2d: (tp, X*Y*Z)
#     sum_squares = np.sum(n_2d**2, axis=0) # sum_squares: (X*Y*Z,)
#     var = sum_squares / np.trace(P) # var: (X*Y*Z,)
#     stds = np.sqrt(var) # stds: (X*Y*Z,)
#     vw_stds = stds.reshape(xdim, ydim, zdim) # vectorwise stds: (X, Y, Z)
#     vw_stds_4d = vw_stds[:, :, :, np.newaxis] # vw_stds_4d: (X, Y, Z, 1)
#     print(vw_stds_4d.shape) # sanity check

#     epsilon = 1e-20 # add a negligible constant to each element of vw_stds_4d to avoid dividing by zero
#     # z_scored = datasets[i] / (vw_stds_4d + epsilon) 
#     z_scored = np.divide(datasets[i], vw_stds_4d, where=vw_stds_4d != 0)
#     print(z_scored.shape) # sanity check, should be (X, Y, Z, components)
#     z_img = nib.Nifti1Image(z_scored, affine=raw_maps[i][1], header=raw_maps[i][2])
#     nib.save(z_img, os.path.join(z_path, raw_maps[i][3] + '_z_scored.nii'))

(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 49)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 58)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 55)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 54)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 52)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 58)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 50)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 50)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 56)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 54)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 51)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 56)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 53)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 55)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 53)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 58)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 57)
(250, 230400)
True
(96, 96, 25, 1)
(96, 96, 25, 56)
